Dataset Split

In [ ]:
import os, shutil, random

# Source paths
BASE_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base"
# Target paths
TARGET_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/code/UltraLight-VM-UNet/data/skin_lesion"

os.makedirs(TARGET_DIR, exist_ok=True)

# 1. Get all image IDs
imgs = [f for f in os.listdir(f"{BASE_DIR}/images") if f.lower().endswith('.jpg')]
random.seed(42) # Fixed random seed for reproducibility
random.shuffle(imgs)

# 2. Split dataset: 80% train, 10% val, 10% test
n = len(imgs)
splits = {
    'train': imgs[:int(n*0.8)],
    'val': imgs[int(n*0.8):int(n*0.9)],
    'test': imgs[int(n*0.9):]
}

# 3. Start copying
for mode, file_list in splits.items():
    print(f"Processing {mode} set, total {len(file_list)} images...")
    os.makedirs(f"{TARGET_DIR}/{mode}/images", exist_ok=True)
    os.makedirs(f"{TARGET_DIR}/{mode}/masks", exist_ok=True)
    
    for f in file_list:
        # Copy image
        shutil.copy(f"{BASE_DIR}/images/{f}", f"{TARGET_DIR}/{mode}/images/{f}")
        # Copy and rename mask: assuming merged mask is named f"{ID}_lesion.png"
        mask_old_name = f.replace(".jpg", "_lesion.png")
        mask_new_name = f.replace(".jpg", ".png")
        shutil.copy(f"{BASE_DIR}/masks_lesion/{mask_old_name}", f"{TARGET_DIR}/{mode}/masks/{mask_new_name}")

print("Dataset construction completed!")

Convert the image and mask data into NumPy array format and save them for use in training the UltraLight-VM-UNet model

In [ ]:
import numpy as np
import cv2
import os
from tqdm import tqdm


height, width = 256, 256
channels = 3

train_number = 3000 
val_number = 400
test_number = 400
total_count = train_number + val_number + test_number

IMG_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base/images"
MSK_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base/masks_lesion"


all_imgs = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.jpg')])[:total_count]

Data_all = np.zeros([total_count, height, width, channels], dtype=np.float64)
Label_all = np.zeros([total_count, height, width], dtype=np.float64)



for idx, fname in enumerate(tqdm(all_imgs)):
    img_path = os.path.join(IMG_DIR, fname)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (width, height), interpolation=cv2.INTER_LINEAR)
    Data_all[idx] = img.astype(np.float64)
    msk_fname = fname.replace(".jpg", "_lesion.png")
    msk_path = os.path.join(MSK_DIR, msk_fname)
    
    mask = cv2.imread(msk_path, 0)
    if mask is None:
        mask = cv2.imread(os.path.join(MSK_DIR, fname.replace(".jpg", ".png")), 0)
        
    mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
    Label_all[idx] = mask.astype(np.float64)


save_dir = "/gpfs/work/bio/yixuanli2204/FYP_Project/code/UltraLight-VM-UNet/data/skin_lesion/"
os.makedirs(save_dir, exist_ok=True)

np.save(save_dir + 'data_train.npy', Data_all[0:train_number])
np.save(save_dir + 'data_val.npy',   Data_all[train_number:train_number+val_number])
np.save(save_dir + 'data_test.npy',  Data_all[train_number+val_number:total_count])

np.save(save_dir + 'mask_train.npy', Label_all[0:train_number])
np.save(save_dir + 'mask_val.npy',   Label_all[train_number:train_number+val_number])
np.save(save_dir + 'mask_test.npy',  Label_all[train_number+val_number:total_count])

print(f"6 files have been saved to: {save_dir}")